In [1]:
import ipdb, traceback
import numpy as np
import math

import pybullet as p

from predicators.settings import CFG, GlobalSettings
from predicators.envs.pybullet_multitable_blocks import PyBulletMultiTableBlocksEnv
from predicators.approaches.oracle_approach import OracleApproach  
from predicators.ground_truth_models import get_gt_options, get_gt_nsrts  
from predicators.structs import GroundAtom, EnvironmentTask, Task, Object  
from predicators.utils import option_plan_to_policy  
from predicators.option_model import create_option_model


CFG.pybullet_robot = "fetch_mobile"

pybullet build time: Jan 29 2025 23:16:28


In [2]:
# 1. Initialize your multi-table environment  
env = PyBulletMultiTableBlocksEnv(use_gui=True, num_tables=3)
CFG.env = env.get_name()

symbolic_robot = env._robot
pybullet_robot = env._pybullet_robot
physics_client_id = env._physics_client_id

INFO:root:'
Proceeding with IK. Passing control to pybullet_helpers/ikfast/utils.py


In [3]:
init_joints = pybullet_robot.get_joints()

In [4]:
table_pose = env._table_poses[0]  # (1.0, -0.5, 0.0)
print(table_pose)

(1.0, -0.5, 0.0)


In [5]:
def draw_circle(
    pid,
    center=(1.0, -0.5, 0.0),
    radius=0.85,
    segments=100,
    color=(1, 0, 0),
    line_width=2,
    life_time=0.0,
):
    ids = []
    for i in range(segments):
        t1 = 2 * math.pi * i / segments
        t2 = 2 * math.pi * (i + 1) / segments
        start = [
            center[0] + radius * math.cos(t1),
            center[1] + radius * math.sin(t1),
            center[2],
        ]
        end = [
            center[0] + radius * math.cos(t2),
            center[1] + radius * math.sin(t2),
            center[2],
        ]
        uid = p.addUserDebugLine(
            start, end, lineColorRGB=color,
            lineWidth=line_width, lifeTime=life_time,
            physicsClientId=pid,
        )
        ids.append(uid)
    return

In [6]:
draw_circle(physics_client_id)

In [7]:
table_configs = {  
    0: {  # Table 0: Random piles  
        'exact_state': {},  
        'setup': 'pile',  
        'params': [2, 3]  # 2 piles, 3 blocks per pile  
    },  
    1: {  # Table 1: Exact pile configuration  
        'exact_state': {  
            'pile1': ['red', 'blue', 'green'],  
            'pile2': ['yellow', 'purple']  
        },  
        'setup': 'exact_pile',  
        'params': None  
    },  
    2: {  # Table 2: Empty initially  
        'exact_state': [],  
        'setup': 'exact_scattered',  
        'params': None  
    }  
}  

In [8]:
# Create the initial state  
initial_state = env.set_state(pybullet_robot, table_configs)

INFO:root:'
Proceeding with IK. Passing control to pybullet_helpers/ikfast/utils.py


In [9]:
pybullet_robot.set_joints(init_joints)

In [14]:
def points_on_circle(x, y, r=0.83, num_points=1):
    """Return points on a circle centered at (x, y) with radius r."""
    angles = np.linspace(0, 2*np.pi, num_points, endpoint=False)
    xs = x + r * np.cos(angles)
    ys = y + r * np.sin(angles)
    return [xs, ys]

In [15]:
# Position robot near the table  
robot_target_x, robot_target_y  = points_on_circle(table_pose[0], table_pose[1])
  
# Move the robot base  
pybullet_robot.move_base_to((robot_target_x, robot_target_y, 0.0),
                            physics_client_id=physics_client_id)

In [16]:
table_center_to_robot_base_coord_diff = (table_pose[0]-robot_target_x, table_pose[1]-robot_target_y)
table_center_to_robot_base_coord_dist = np.linalg.norm(table_center_to_robot_base_coord_diff)

In [17]:
table_center_to_robot_base_coord_dist

0.8300000000000001

In [18]:
robot_target_x

1.2

In [19]:
robot_target_y

0.30000000000000004

In [10]:
# Get the updated state after robot movement  
updated_initial_state = env._get_state()

In [11]:
updated_initial_state

PyBulletState(data={robby:robot: array([ 2.2       , -0.29999998,  0.70000005,  1.        ], dtype=float32), block0_0_0:block: array([ 0.9939536 , -0.61641663,  0.2225    ,  0.        ,  0.89360166,
        0.7261957 ,  0.65861356], dtype=float32), block0_0_1:block: array([ 0.9939536 , -0.61641663,  0.2675    ,  0.        ,  0.792373  ,
        0.18619148,  0.3466251 ], dtype=float32), block0_0_2:block: array([ 0.9939536 , -0.61641663,  0.3125    ,  0.        ,  0.6922928 ,
        0.99513525,  0.7589759 ], dtype=float32), block0_1_0:block: array([ 0.92468613, -0.3479963 ,  0.2225    ,  0.        ,  0.3637625 ,
        0.9078099 ,  0.2009277 ], dtype=float32), block0_1_1:block: array([ 0.92468613, -0.3479963 ,  0.2675    ,  0.        ,  0.9774682 ,
        0.3761039 ,  0.4638067 ], dtype=float32), block0_1_2:block: array([ 0.92468613, -0.3479963 ,  0.3125    ,  0.        ,  0.34069642,
        0.02192815,  0.19479412], dtype=float32), block1_1_0:block: array([-1.6713777e+00,  1.6158205